# LDPC BpLsdDecoder Order Dependence Analysis

This notebook analyzes whether the upstream `ldpc` package has order-dependent behavior due to a known bug.

**Bug details:**
- In ldpc's `_bplsd_decoder.pyx`, `zero_syndrome` is not initialized before use
- With Cython's `initializedcheck=False`, this causes undefined behavior
- Internal state (`bp_decoding`, `statistics`) may leak between `decode()` calls

**Tests:**
1. **Zero-syndrome consistency**: Repeated decodes of all-zero syndrome should be identical
2. **State leakage**: `bp_decoding` should reset after decoding zero-syndrome
3. **Order dependence**: Same syndrome → same result regardless of decode order
4. **Fixed-class decoder**: Cached decoder should not introduce order dependence

## 1. Configuration

Configure the analysis parameters below, then run the cell to generate the script command.

In [1]:
# ==============================================================================
# CONFIGURATION - Set your analysis parameters here
# ==============================================================================

# Code parameters
N_QUBITS = 144           # BB code variant: 72, 90, 108, 144, 288, 360, 756
P = 0.003                # Physical error rate

# Test parameters
N_SHOTS = 1000           # Number of shots for tests 2, 3, 4
N_REPEATS = 20          # Number of repeats for test 1 (zero syndrome consistency)
SEED = 42               # Random seed

# Output path (auto-generated based on parameters)
OUTPUT_DIR = "simulations/data/ldpc_order_dependence"

In [3]:
# ==============================================================================
# Generate script command based on configuration
# ==============================================================================

from pathlib import Path

# Generate output filename
OUTPUT_FILENAME = f"bb_n{N_QUBITS}_p{P}_shots{N_SHOTS}_repeats{N_REPEATS}.parquet"
OUTPUT_PATH = Path(OUTPUT_DIR) / OUTPUT_FILENAME

# Build command
SCRIPT_PATH = "simulations/analysis/scripts/run_ldpc_order_dependence_analysis.py"

cmd_parts = [
    f"python {SCRIPT_PATH}",
    f"    --n-qubits {N_QUBITS}",
    f"    --p {P}",
    f"    --n-shots {N_SHOTS}",
    f"    --n-repeats {N_REPEATS}",
    f"    --seed {SEED}",
    f"    --output {OUTPUT_PATH}",
]

command = " \\\n".join(cmd_parts)

# Display
print("=" * 70)
print("GENERATED COMMAND")
print("=" * 70)
print()
print(command)
print()
print("=" * 70)
print(f"Output file: {OUTPUT_PATH}")
print("=" * 70)

# Store for later use
RESULTS_PATH = Path("../../..")
RESULTS_PATH = RESULTS_PATH / OUTPUT_PATH

GENERATED COMMAND

python simulations/analysis/scripts/run_ldpc_order_dependence_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 1000 \
    --n-repeats 20 \
    --seed 42 \
    --output simulations/data/ldpc_order_dependence/bb_n144_p0.003_shots1000_repeats20.parquet

Output file: simulations/data/ldpc_order_dependence/bb_n144_p0.003_shots1000_repeats20.parquet


In [ ]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

## 2. Load Results

In [ ]:
# ==============================================================================
# Load results (uses RESULTS_PATH from configuration above)
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first using the command above.")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_parquet(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

print(f"\nTest counts by type:")
print(df_results["test_name"].value_counts())

## 3. Test 1: Zero-Syndrome Consistency

Repeated decodes of all-zero syndrome should produce identical results. Any variation indicates the uninitialized `zero_syndrome` bug.

In [ ]:
# ==============================================================================
# Test 1 Analysis: Zero-Syndrome Consistency
# ==============================================================================

df_test1 = df_results[df_results["test_name"] == "zero_syndrome_consistency"].copy()

if len(df_test1) == 0:
    print("No Test 1 results found.")
else:
    print("=" * 60)
    print("TEST 1: ZERO-SYNDROME CONSISTENCY")
    print("=" * 60)
    
    n_iterations = len(df_test1)
    n_match_first = df_test1["match_first"].sum()
    
    print(f"\nIterations: {n_iterations}")
    print(f"Match first result: {n_match_first}/{n_iterations} ({100*n_match_first/n_iterations:.1f}%)")
    
    # Check for variation in key metrics
    print(f"\nMetric variations:")
    print(f"  pred_hash unique values: {df_test1['pred_hash'].nunique()}")
    print(f"  pred_bp_hash unique values: {df_test1['pred_bp_hash'].nunique()}")
    print(f"  pred_llr range: [{df_test1['pred_llr'].min():.6f}, {df_test1['pred_llr'].max():.6f}]")
    print(f"  bp_decoding_hash unique values: {df_test1['bp_decoding_hash'].nunique()}")
    
    # Result
    test1_pass = (n_match_first == n_iterations)
    print(f"\nResult: {'PASS' if test1_pass else 'FAIL'}")
    if not test1_pass:
        print("WARNING: Inconsistent results detected! Bug is likely present.")

In [ ]:
# ==============================================================================
# Test 1 Visualization
# ==============================================================================

if len(df_test1) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("pred_llr Across Iterations", "Hash Distribution")
    )
    
    # Left: pred_llr across iterations
    fig.add_trace(
        go.Scatter(
            x=df_test1["iteration"],
            y=df_test1["pred_llr"],
            mode="lines+markers",
            name="pred_llr",
            marker=dict(size=8),
        ),
        row=1, col=1,
    )
    
    # Right: Hash uniqueness (bar chart)
    hash_counts = {
        "pred_hash": df_test1["pred_hash"].nunique(),
        "pred_bp_hash": df_test1["pred_bp_hash"].nunique(),
        "bp_decoding_hash": df_test1["bp_decoding_hash"].nunique(),
    }
    
    fig.add_trace(
        go.Bar(
            x=list(hash_counts.keys()),
            y=list(hash_counts.values()),
            marker_color=["green" if v == 1 else "red" for v in hash_counts.values()],
            text=[f"{v}" for v in hash_counts.values()],
            textposition="outside",
        ),
        row=1, col=2,
    )
    
    # Add horizontal line at 1 (expected)
    fig.add_hline(y=1, line_dash="dash", line_color="green", row=1, col=2)
    
    fig.update_xaxes(title_text="Iteration", row=1, col=1)
    fig.update_yaxes(title_text="pred_llr", row=1, col=1)
    fig.update_xaxes(title_text="Hash Type", row=1, col=2)
    fig.update_yaxes(title_text="Unique Values (expect 1)", row=1, col=2)
    
    fig.update_layout(
        title="Test 1: Zero-Syndrome Consistency",
        height=400,
        showlegend=False,
    )
    
    fig.show()

## 4. Test 2: State Leakage

After decoding a non-trivial syndrome, then decoding all-zero syndrome, the internal `bp_decoding` should be properly reset.

In [ ]:
# ==============================================================================
# Test 2 Analysis: State Leakage
# ==============================================================================

df_test2 = df_results[df_results["test_name"] == "state_leakage"].copy()

if len(df_test2) == 0:
    print("No Test 2 results found.")
else:
    print("=" * 60)
    print("TEST 2: STATE LEAKAGE")
    print("=" * 60)
    
    n_shots = len(df_test2)
    n_bp_leaked = df_test2["bp_decoding_leaked"].sum()
    n_stats_leaked = df_test2["statistics_leaked"].sum()
    
    print(f"\nShots tested: {n_shots}")
    print(f"bp_decoding leaked: {n_bp_leaked}/{n_shots} ({100*n_bp_leaked/n_shots:.1f}%)")
    print(f"statistics leaked: {n_stats_leaked}/{n_shots} ({100*n_stats_leaked/n_shots:.1f}%)")
    
    # Zero syndrome results
    print(f"\nZero syndrome prediction:")
    print(f"  pred_sum range: [{df_test2['zero_pred_sum'].min()}, {df_test2['zero_pred_sum'].max()}]")
    print(f"  pred_bp_sum range: [{df_test2['zero_pred_bp_sum'].min()}, {df_test2['zero_pred_bp_sum'].max()}]")
    
    # Result
    test2_pass = (n_bp_leaked == 0)
    print(f"\nResult: {'PASS' if test2_pass else 'FAIL'}")
    if not test2_pass:
        print("WARNING: State leakage detected! Internal state not properly reset.")

In [ ]:
# ==============================================================================
# Test 2 Visualization
# ==============================================================================

if len(df_test2) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Non-Trivial vs Zero Syndrome LLR",
            "State Leakage by Non-Trivial Violations"
        )
    )
    
    # Left: LLR comparison
    fig.add_trace(
        go.Scatter(
            x=df_test2["nontrivial_pred_llr"],
            y=df_test2["zero_pred_llr"],
            mode="markers",
            marker=dict(
                size=6,
                color=df_test2["bp_decoding_leaked"].astype(int),
                colorscale=[[0, "green"], [1, "red"]],
                opacity=0.6,
            ),
            hovertemplate="Non-trivial LLR: %{x:.2f}<br>Zero LLR: %{y:.2f}<extra></extra>",
        ),
        row=1, col=1,
    )
    
    # Right: Leakage by violation count
    df_test2["violations_bin"] = pd.cut(
        df_test2["nontrivial_violations"],
        bins=[0, 5, 10, 20, 50, 100, float("inf")],
        labels=["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
    )
    
    leak_by_bin = df_test2.groupby("violations_bin", observed=True).agg({
        "bp_decoding_leaked": ["sum", "count"]
    })
    leak_by_bin.columns = ["leaked", "total"]
    leak_by_bin["leak_rate"] = leak_by_bin["leaked"] / leak_by_bin["total"]
    
    fig.add_trace(
        go.Bar(
            x=leak_by_bin.index.astype(str),
            y=leak_by_bin["leak_rate"] * 100,
            marker_color="steelblue",
            text=[f"{r:.1f}%" for r in leak_by_bin["leak_rate"] * 100],
            textposition="outside",
        ),
        row=1, col=2,
    )
    
    fig.update_xaxes(title_text="Non-Trivial pred_llr", row=1, col=1)
    fig.update_yaxes(title_text="Zero Syndrome pred_llr", row=1, col=1)
    fig.update_xaxes(title_text="Violation Count Bin", row=1, col=2)
    fig.update_yaxes(title_text="Leakage Rate (%)", row=1, col=2)
    
    fig.update_layout(
        title="Test 2: State Leakage (red = leaked)",
        height=400,
        showlegend=False,
    )
    
    fig.show()

## 5. Test 3: Order Dependence

Using two separate decoders, we decode syndromes in different orders. Same syndrome should produce same result regardless of order.

In [ ]:
# ==============================================================================
# Test 3 Analysis: Order Dependence
# ==============================================================================

df_test3 = df_results[df_results["test_name"] == "order_dependence"].copy()

if len(df_test3) == 0:
    print("No Test 3 results found.")
else:
    print("=" * 60)
    print("TEST 3: ORDER DEPENDENCE")
    print("=" * 60)
    
    n_pairs = len(df_test3)
    n_all_match = df_test3["all_match"].sum()
    
    print(f"\nSyndrome pairs tested: {n_pairs}")
    print(f"All results match: {n_all_match}/{n_pairs} ({100*n_all_match/n_pairs:.1f}%)")
    
    # Breakdown by syndrome
    n_a_pred_mismatch = (~df_test3["a_pred_match"]).sum()
    n_a_bp_mismatch = (~df_test3["a_pred_bp_match"]).sum()
    n_a_llr_mismatch = (~df_test3["a_llr_match"]).sum()
    n_b_pred_mismatch = (~df_test3["b_pred_match"]).sum()
    n_b_bp_mismatch = (~df_test3["b_pred_bp_match"]).sum()
    n_b_llr_mismatch = (~df_test3["b_llr_match"]).sum()
    
    print(f"\nSyndrome A mismatches:")
    print(f"  pred: {n_a_pred_mismatch}")
    print(f"  pred_bp: {n_a_bp_mismatch}")
    print(f"  pred_llr: {n_a_llr_mismatch}")
    
    print(f"\nSyndrome B mismatches:")
    print(f"  pred: {n_b_pred_mismatch}")
    print(f"  pred_bp: {n_b_bp_mismatch}")
    print(f"  pred_llr: {n_b_llr_mismatch}")
    
    # LLR differences
    print(f"\nLLR difference statistics:")
    print(f"  Syndrome A: mean={df_test3['a_llr_diff'].mean():.6f}, std={df_test3['a_llr_diff'].std():.6f}")
    print(f"  Syndrome B: mean={df_test3['b_llr_diff'].mean():.6f}, std={df_test3['b_llr_diff'].std():.6f}")
    
    # Result
    test3_pass = (n_all_match == n_pairs)
    print(f"\nResult: {'PASS' if test3_pass else 'FAIL'}")
    if not test3_pass:
        print("WARNING: Order-dependent behavior detected!")

In [ ]:
# ==============================================================================
# Test 3 Visualization
# ==============================================================================

if len(df_test3) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "LLR Difference Distribution (Syndrome A)",
            "LLR Difference Distribution (Syndrome B)"
        )
    )
    
    # Left: Syndrome A LLR differences
    fig.add_trace(
        go.Histogram(
            x=df_test3["a_llr_diff"],
            nbinsx=50,
            marker_color="steelblue",
            opacity=0.7,
        ),
        row=1, col=1,
    )
    
    # Right: Syndrome B LLR differences
    fig.add_trace(
        go.Histogram(
            x=df_test3["b_llr_diff"],
            nbinsx=50,
            marker_color="coral",
            opacity=0.7,
        ),
        row=1, col=2,
    )
    
    # Add vertical lines at 0
    fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
    fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=2)
    
    fig.update_xaxes(title_text="LLR Difference (Order1 - Order2)", row=1, col=1)
    fig.update_xaxes(title_text="LLR Difference (Order1 - Order2)", row=1, col=2)
    fig.update_yaxes(title_text="Count", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=2)
    
    fig.update_layout(
        title="Test 3: Order Dependence - LLR Differences (expect all zero)",
        height=400,
        showlegend=False,
    )
    
    fig.show()

## 6. Test 4: Fixed-Class Decoder Order Dependence

This test is critical for gap proxy methods. The cached `_decoder_for_fixed_class` is reused for all logical class explorations.

In [ ]:
# ==============================================================================
# Test 4 Analysis: Fixed-Class Decoder Order Dependence
# ==============================================================================

df_test4 = df_results[df_results["test_name"] == "fixed_class_order"].copy()

if len(df_test4) == 0:
    print("No Test 4 results found.")
else:
    print("=" * 60)
    print("TEST 4: FIXED-CLASS DECODER ORDER DEPENDENCE")
    print("=" * 60)
    
    n_shots = len(df_test4)
    n_all_match = df_test4["all_match"].sum()
    n_class0_mismatch = (~df_test4["class0_llr_match"]).sum()
    n_class1_mismatch = (~df_test4["class1_llr_match"]).sum()
    
    print(f"\nShots tested: {n_shots}")
    print(f"All results match: {n_all_match}/{n_shots} ({100*n_all_match/n_shots:.1f}%)")
    print(f"\nClass 0 LLR mismatches: {n_class0_mismatch}/{n_shots}")
    print(f"Class 1 LLR mismatches: {n_class1_mismatch}/{n_shots}")
    
    # LLR differences
    print(f"\nLLR difference statistics:")
    print(f"  Class 0: mean={df_test4['class0_llr_diff'].mean():.6f}, std={df_test4['class0_llr_diff'].std():.6f}")
    print(f"  Class 1: mean={df_test4['class1_llr_diff'].mean():.6f}, std={df_test4['class1_llr_diff'].std():.6f}")
    
    # Result
    test4_pass = (n_all_match == n_shots)
    print(f"\nResult: {'PASS' if test4_pass else 'FAIL'}")
    if not test4_pass:
        print("WARNING: Fixed-class decoder order dependence detected!")
        print("This affects gap proxy method comparisons!")

In [ ]:
# ==============================================================================
# Test 4 Visualization
# ==============================================================================

if len(df_test4) > 0:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "Class 0: LLR Order1 vs Order2",
            "Class 1: LLR Order1 vs Order2"
        )
    )
    
    # Left: Class 0
    fig.add_trace(
        go.Scatter(
            x=df_test4["class0_llr_order1"],
            y=df_test4["class0_llr_order2"],
            mode="markers",
            marker=dict(
                size=6,
                color=df_test4["class0_llr_match"].map({True: "green", False: "red"}),
                opacity=0.6,
            ),
        ),
        row=1, col=1,
    )
    
    # Right: Class 1
    fig.add_trace(
        go.Scatter(
            x=df_test4["class1_llr_order1"],
            y=df_test4["class1_llr_order2"],
            mode="markers",
            marker=dict(
                size=6,
                color=df_test4["class1_llr_match"].map({True: "green", False: "red"}),
                opacity=0.6,
            ),
        ),
        row=1, col=2,
    )
    
    # Add diagonal lines (y=x)
    for col in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=[df_test4[["class0_llr_order1", "class1_llr_order1"]].min().min(),
                   df_test4[["class0_llr_order1", "class1_llr_order1"]].max().max()],
                y=[df_test4[["class0_llr_order1", "class1_llr_order1"]].min().min(),
                   df_test4[["class0_llr_order1", "class1_llr_order1"]].max().max()],
                mode="lines",
                line=dict(color="black", dash="dash"),
                showlegend=False,
            ),
            row=1, col=col,
        )
    
    fig.update_xaxes(title_text="LLR (Order 1: class0→class1)", row=1, col=1)
    fig.update_yaxes(title_text="LLR (Order 2: class1→class0)", row=1, col=1)
    fig.update_xaxes(title_text="LLR (Order 1: class0→class1)", row=1, col=2)
    fig.update_yaxes(title_text="LLR (Order 2: class1→class0)", row=1, col=2)
    
    fig.update_layout(
        title="Test 4: Fixed-Class Decoder (expect all on diagonal)",
        height=450,
        showlegend=False,
    )
    
    fig.show()

## 7. Summary

In [ ]:
# ==============================================================================
# Overall Summary
# ==============================================================================

print("=" * 60)
print("OVERALL SUMMARY")
print("=" * 60)

# Test 1
df_test1 = df_results[df_results["test_name"] == "zero_syndrome_consistency"]
if len(df_test1) > 0:
    test1_pass = df_test1["match_first"].all()
else:
    test1_pass = None

# Test 2
df_test2 = df_results[df_results["test_name"] == "state_leakage"]
if len(df_test2) > 0:
    test2_pass = not df_test2["bp_decoding_leaked"].any()
else:
    test2_pass = None

# Test 3
df_test3 = df_results[df_results["test_name"] == "order_dependence"]
if len(df_test3) > 0:
    test3_pass = df_test3["all_match"].all()
else:
    test3_pass = None

# Test 4
df_test4 = df_results[df_results["test_name"] == "fixed_class_order"]
if len(df_test4) > 0:
    test4_pass = df_test4["all_match"].all()
else:
    test4_pass = None

# Print results table
results_table = [
    ("Test 1: Zero-syndrome consistency", test1_pass),
    ("Test 2: State leakage", test2_pass),
    ("Test 3: Order dependence", test3_pass),
    ("Test 4: Fixed-class decoder", test4_pass),
]

print()
for test_name, passed in results_table:
    if passed is None:
        status = "SKIPPED"
        symbol = "-"
    elif passed:
        status = "PASS"
        symbol = "✓"
    else:
        status = "FAIL"
        symbol = "✗"
    print(f"  {symbol} {test_name}: {status}")

# Overall
all_results = [r for _, r in results_table if r is not None]
if all_results:
    all_pass = all(all_results)
    print()
    if all_pass:
        print("CONCLUSION: All tests PASSED. No order-dependent bug detected.")
        print("The decoder caching approach is safe to use.")
    else:
        print("CONCLUSION: Some tests FAILED. Order-dependent behavior detected!")
        print("")
        print("Recommendations:")
        print("  1. Report upstream to ldpc maintainers")
        print("  2. Create fresh decoder instances per decode (workaround)")
        print("  3. Patch ldpc locally (initialize zero_syndrome = True)")